<a href="https://colab.research.google.com/github/venkatasai-eng/MLA0305-REINFORCEMENT-LEARNING-/blob/main/EXP_NO_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)

states = 5
actions = 2
episodes = 100
gamma = 0.9
clip = 0.2

def step(s, a):
    ns = min(s + 1, 4) if a == 1 else max(s - 1, 0)
    r = 10 if ns == 4 else -1
    return ns, r, ns == 4

def encode(s):
    x = np.zeros(states)
    x[s] = 1
    return x

inp = layers.Input(shape=(states,))
x = layers.Dense(16, activation="relu")(inp)
x = layers.Dense(16, activation="relu")(x)

actor = layers.Dense(actions, activation="softmax")(x)
critic = layers.Dense(1)(x)

model = tf.keras.Model(inp, [actor, critic])
optimizer = tf.keras.optimizers.Adam(0.001)

for episode in range(episodes):

    s = 0
    states_list = []
    actions_list = []
    rewards_list = []
    old_probs = []

    while True:

        x = encode(s).reshape(1, -1)

        prob, value = model(x)

        p = prob[0].numpy()
        a = np.random.choice(actions, p=p)

        ns, r, done = step(s, a)

        states_list.append(encode(s))
        actions_list.append(a)
        rewards_list.append(r)
        old_probs.append(p[a])

        s = ns

        if done:
            break

    returns = []
    G = 0

    for r in reversed(rewards_list):
        G = r + gamma * G
        returns.insert(0, G)

    states_tensor = tf.convert_to_tensor(
        np.array(states_list),
        dtype=tf.float32
    )

    actions_tensor = tf.convert_to_tensor(
        actions_list,
        dtype=tf.int32
    )

    returns_tensor = tf.convert_to_tensor(
        returns,
        dtype=tf.float32
    )

    old_probs_tensor = tf.convert_to_tensor(
        old_probs,
        dtype=tf.float32
    )

    with tf.GradientTape() as tape:

        probs, values = model(states_tensor)

        selected_probs = tf.gather(
            probs,
            actions_tensor,
            axis=1,
            batch_dims=1
        )

        ratio = selected_probs / (
            old_probs_tensor + 1e-8
        )

        advantage = returns_tensor - tf.stop_gradient(
            tf.squeeze(values)
        )

        clipped_ratio = tf.clip_by_value(
            ratio,
            1 - clip,
            1 + clip
        )

        actor_loss = -tf.reduce_mean(
            tf.minimum(
                ratio * advantage,
                clipped_ratio * advantage
            )
        )

        critic_loss = tf.reduce_mean(
            tf.square(
                returns_tensor - tf.squeeze(values)
            )
        )

        loss = actor_loss + 0.5 * critic_loss

    gradients = tape.gradient(
        loss,
        model.trainable_variables
    )

    optimizer.apply_gradients(
        zip(gradients, model.trainable_variables)
    )

    if (episode + 1) % 20 == 0:
        print(
            "Episode:",
            episode + 1,
            "Reward:",
            sum(rewards_list)
        )

print("\nPPO Training Completed")

print("\nLearned Policy:")

for s in range(states):

    p, v = model(
        encode(s).reshape(1, -1)
    )

    print(
        "State:",
        s,
        "Action:",
        np.argmax(p[0].numpy()),
        "Probability:",
        np.round(p[0].numpy(), 3),
        "Value:",
        round(float(v[0, 0]), 2)
    )

Episode: 20 Reward: -15
Episode: 40 Reward: -11
Episode: 60 Reward: -9
Episode: 80 Reward: 3
Episode: 100 Reward: 7

PPO Training Completed

Learned Policy:
State: 0 Action: 1 Probability: [0.327 0.673] Value: -0.01
State: 1 Action: 1 Probability: [0.31 0.69] Value: 0.06
State: 2 Action: 1 Probability: [0.37 0.63] Value: 0.14
State: 3 Action: 1 Probability: [0.359 0.641] Value: 0.15
State: 4 Action: 1 Probability: [0.302 0.698] Value: 0.05
